# 02 · 핵심 분석: 교차표·집단비교·회귀  (모듈 3)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amnotyoung/oda-ai-stats/blob/main/notebooks/02_core_analysis.ipynb)

> 대표 분석 3종을 Python으로 수행하고, **같은 분석을 STATA로도**(각 절 🔁) 해서
> 두 결과의 숫자가 같은지 **교차검증**한다. — "숫자는 같다. 출력 모양만 다르다."

In [ ]:
import pandas as pd, numpy as np
from scipy import stats
import statsmodels.formula.api as smf
df = pd.read_csv("https://raw.githubusercontent.com/amnotyoung/oda-ai-stats/main/data/wdi_panel.csv")
df["log_gdp"] = np.log(df["gdp_pc"])
df["log_pop"] = np.log(df["pop"])
print("준비 완료:", df.shape)

## 1) 교차표 — 지역 × 소득그룹 (국가 수)
지역·소득은 국가 속성이므로 **국가 단위로** 집계한다.

In [ ]:
countries = df.drop_duplicates("economy")
pd.crosstab(countries["region_name"], countries["income_name"])

🔁 **STATA**: `tabulate region_name income_name`  ·  코드 → `stata/02_crosstab.do`

## 2) 집단 비교 — 지역 격차(t검정), 소득그룹 차이(ANOVA)

In [ ]:
# t검정: 사하라이남 아프리카 vs 그 외 (기대수명) — 분산이 달라 Welch 사용
ssa  = df.loc[df.region_name == "Sub-Saharan Africa", "life_exp"].dropna()
rest = df.loc[df.region_name != "Sub-Saharan Africa", "life_exp"].dropna()
t, p = stats.ttest_ind(ssa, rest, equal_var=False)   # Welch → STATA는 ttest ..., unequal
print(f"[t검정] 사하라이남={ssa.mean():.1f}세  그외={rest.mean():.1f}세  t={t:.1f}  p={p:.1e}")

# ANOVA: 소득그룹 간 기대수명
groups = [g["life_exp"].dropna().values for _, g in df.groupby("income_name")]
F, p2 = stats.f_oneway(*[g for g in groups if len(g) > 1])
print(f"[ANOVA] 소득그룹별 기대수명  F={F:.0f}  p={p2:.1e}")

🔁 **STATA**: `ttest life_exp, by(ssa) unequal`  ·  `oneway life_exp income_n`  ·  코드 → `stata/03_group_compare.do`

## 3) 회귀분석 — Preston 곡선 (소득 → 기대수명)
개발경제의 고전. 1인당 GDP는 왜도가 크므로 **로그**를 취한다 — AI가 자동으로 안 할 수 있는 **설계 판단**.

In [ ]:
m = smf.ols("life_exp ~ log_gdp", data=df).fit()
print(m.summary().tables[1])
print(f"\nR² = {m.rsquared:.3f}")

**해석**: `log_gdp` 계수가 **양수** → 소득이 높을수록 기대수명↑. (소득 e배 ≈ +계수 만큼 수명)
R²가 높아 소득만으로 기대수명 차이의 상당 부분이 설명된다.

🔁 **STATA**: `regress life_exp log_gdp` — **한 줄**.  코드 → `stata/04_regression.do`
> 양쪽 계수·R²가 같은지 **교차검증**: 같으면 신뢰, 다르면 하나가 틀린 것.

---
✅ **핵심**: 교차표·검정·회귀를 입문자가 첫날 직접 돌렸다. 비결은 *문법 암기*가 아니라
*무엇을·왜 분석할지 설계*하고 *검증·해석*하는 것.